## 04. Modeliranje: Eksperiment A (SC + classical FE + ML)

### Uvoz biblioteka

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

### Učitavanje podataka

In [11]:
DATA_DIR = "../../data/processed/localization"
FEATURES_DIR = "../../data/features/localization"
MODELS_DIR = "../../data/models/localization"
METRICS_DIR = "../../results/metrics/localization"

os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42

# Train/test split (Entry + enkodirana labela) 
train_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_entries.csv"))
test_entries_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_entries.csv"))

entries_sc_train = train_entries_df["Entry"].values
entries_sc_test = test_entries_df["Entry"].values

y_sc_train = train_entries_df["label_encoded"].values
y_sc_test = test_entries_df["label_encoded"].values

# LabelEncoder fit-ovan jednom u 03 (nema ponovnog fitovanja)
le = joblib.load(os.path.join(FEATURES_DIR, "sc_label_encoder.pkl"))

print(f"Train skup: {len(entries_sc_train)} proteina")
print(f"Test skup:  {len(entries_sc_test)} proteina")
print(f"Klase: {list(le.classes_)}")

Train skup: 8487 proteina
Test skup:  2122 proteina
Klase: ['Cell membrane', 'Cytoplasm', 'Mitochondrion', 'Nucleus', 'Secreted']


### Učitavanje raw obilježja

In [3]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")
aac_cols = list(aac_df.columns)

print(f"AAC-CV kolone: {len(aac_cols)}")
aac_df.head()

AAC-CV kolone: 20


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
A0A0C4DH62,0.058824,0.000000,0.000000,0.058824,0.058824,0.117647,0.058824,0.000000,0.000000,0.058824,0.000000,0.000000,0.000000,0.117647,0.000000,0.117647,0.117647,0.117647,0.058824,0.058824
A0A0A0MT87,0.125000,0.000000,0.000000,0.000000,0.062500,0.125000,0.000000,0.062500,0.062500,0.125000,0.000000,0.062500,0.000000,0.062500,0.062500,0.062500,0.062500,0.062500,0.000000,0.062500
P01880,0.074419,0.018605,0.034884,0.062791,0.027907,0.046512,0.020930,0.023256,0.048837,0.097674,0.011628,0.023256,0.095349,0.053488,0.048837,0.100000,0.097674,0.065116,0.025581,0.023256
P01876,0.062814,0.035176,0.032663,0.052764,0.035176,0.060302,0.017588,0.010050,0.030151,0.115578,0.005025,0.022613,0.115578,0.042714,0.035176,0.103015,0.118090,0.065327,0.020101,0.020101
P01877,0.066496,0.038363,0.035806,0.056266,0.033248,0.061381,0.020460,0.010230,0.030691,0.112532,0.007673,0.025575,0.104859,0.046036,0.038363,0.092072,0.109974,0.069054,0.020460,0.020460


In [4]:
# Sirove sekvence (fitovanje TF-IDF+SVD dešava se unutar Pipeline-a, po CV foldu)
sc_train_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_sequences_raw.csv"), index_col="Entry")
sc_test_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_sequences_raw.csv"), index_col="Entry")
seq_col = "Sequence"

print(f"Sirove sekvence — train: {sc_train_seq_df.shape[0]} | test: {sc_test_seq_df.shape[0]}")
sc_train_seq_df.head()

Sirove sekvence — train: 8487 | test: 2122


,Sequence
Entry,
Q04323,MAELTALESLIEMGFPRGRAEKALALTGNQGIEAAMDWLMEHEDDP...
Q32ZL2,MPLLPAALTSSMLYFQMVIMAGTVMLAYYFEYTDTFTVNVQGFFCH...
Q9NQW7,MPPKVTSELLRQLRQAMRNSEYVTEPIQAYIIPSGDAHQSEYIAPC...
Q9HA72,MAALIAENFRFLSLFFKSKDVMIFNGLVALGTVGSQELFSVVAFHC...
P45954,MEGLAVRLLRGSRLLRRNFLTCLSSWKIPPHVSKSSQSEALLNITN...


In [5]:
# Sirove (neskalirane) physchem osobine (StandardScaler fituje se unutar Pipeline-a, po CV foldu)
sc_train_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_train_physchem_raw.csv"), index_col="Entry")
sc_test_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "sc_test_physchem_raw.csv"), index_col="Entry")
ph_cols = list(sc_train_ph_df.columns)

print(f"PH kolone: {ph_cols}")
sc_train_ph_df.head()

PH kolone: ['MW', 'pI', 'GRAVY', 'Aromaticity', 'Instability']


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
Q04323,33324.8966,5.225004,-0.991246,0.026936,70.349495
Q32ZL2,35427.0317,6.583911,0.378816,0.109034,29.575078
Q9NQW7,69917.0048,5.423656,-0.245907,0.089888,37.764366
Q9HA72,36174.2020,7.613151,0.117337,0.114551,39.391641
P45954,47484.8712,6.529175,-0.128472,0.087963,34.800694


### Sastavljanje DataFrame-a 

Spajamo AAC-CV, raw sekvence i neskalirane PH vrijednosti u jedan DataFrame po Entry-ju, poravnat sa redoslijedom labela (y_sc_train/y_sc_test).

In [6]:
X_train_raw = aac_df.join(sc_train_seq_df, how="inner").join(sc_train_ph_df, how="inner") 
X_test_raw = aac_df.join(sc_test_seq_df, how="inner").join(sc_test_ph_df, how="inner") 

# Poravnanje redoslijeda redova sa redoslijedom labela
X_train_raw = X_train_raw.loc[entries_sc_train]
X_test_raw = X_test_raw.loc[entries_sc_test]

assert list(X_train_raw.index) == list(entries_sc_train)
assert list(X_test_raw.index) == list(entries_sc_test)

print(f"X_train_raw: {X_train_raw.shape} | X_test_raw: {X_test_raw.shape}")
X_train_raw.head()

X_train_raw: (8487, 26) | X_test_raw: (2122, 26)


,A,C,D,E,F,G,H,I,K,L,...,T,V,W,Y,Sequence,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,,,,,,,,,,,,,,,,,
Q04323,0.097643,0.006734,0.037037,0.158249,0.013468,0.080808,0.010101,0.020202,0.040404,0.097643,...,0.030303,0.043771,0.003367,0.010101,MAELTALESLIEMGFPRGRAEKALALTGNQGIEAAMDWLMEHEDDP...,33324.8966,5.225004,-0.991246,0.026936,70.349495
Q32ZL2,0.096573,0.031153,0.028037,0.043614,0.059190,0.059190,0.018692,0.056075,0.031153,0.102804,...,0.077882,0.093458,0.003115,0.046729,MPLLPAALTSSMLYFQMVIMAGTVMLAYYFEYTDTFTVNVQGFFCH...,35427.0317,6.583911,0.378816,0.109034,29.575078
Q9NQW7,0.073836,0.019262,0.067416,0.070626,0.033708,0.064205,0.028892,0.060995,0.059390,0.091493,...,0.064205,0.069021,0.020867,0.035313,MPPKVTSELLRQLRQAMRNSEYVTEPIQAYIIPSGDAHQSEYIAPC...,69917.0048,5.423656,-0.245907,0.089888,37.764366
Q9HA72,0.111455,0.024768,0.024768,0.061920,0.061920,0.052632,0.024768,0.043344,0.024768,0.126935,...,0.034056,0.074303,0.018576,0.034056,MAALIAENFRFLSLFFKSKDVMIFNGLVALGTVGSQELFSVVAFHC...,36174.2020,7.613151,0.117337,0.114551,39.391641
P45954,0.087963,0.013889,0.034722,0.071759,0.041667,0.087963,0.020833,0.071759,0.067130,0.097222,...,0.062500,0.050926,0.006944,0.039352,MEGLAVRLLRGSRLLRRNFLTCLSSWKIPPHVSKSSQSEALLNITN...,47484.8712,6.529175,-0.128472,0.087963,34.800694


### Definisanje feature-ser kombinacija (kroz ColumnTransfer)

In [7]:
# Funkcija koja bira odgovarajuci transformator i vraca Pipeline
# (svaka feature-set/model kombinacija ima sopstveni transformator)
def make_tfidf_svd_step():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=3000, lowercase=False)),
        ("svd", TruncatedSVD(n_components=150, random_state=RANDOM_STATE)),
    ])

# Funkcija koja vraca ColumnTransfer za odgovarajucu feature-set kombinaciju
def make_preprocessor(feature_set_name):
    if feature_set_name == "AAC-CV":
        transformers = [("aac", "passthrough", aac_cols)]
    elif feature_set_name == "TFIDF-SVD":
        transformers = [("tfidf-svd", make_tfidf_svd_step(), seq_col)]
    elif feature_set_name == "PH":
        transformers = [("ph_scaled", StandardScaler(), ph_cols)]
    elif feature_set_name == "AAC-CV+TFIDF-SVD":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
        ]
    elif feature_set_name == "AAC-CV+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "TFIDF-SVD+PH":
        transformers = [
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "AAC-CV+TFIDF-SVD+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    else:
        raise ValueError(f"Nepoznat feature set: {feature_set_name}")

    return ColumnTransformer(transformers=transformers, remainder="drop")

FEATURE_SET_NAMES = [
    "AAC-CV",
    "TFIDF-SVD",
    "PH",
    "AAC-CV+TFIDF-SVD",
    "AAC-CV+PH",
    "TFIDF-SVD+PH",
    "AAC-CV+TFIDF-SVD+PH",
]


### AAC-CV 

In [ ]:
'''# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
# Selektujemo redove koji pripadaju SC train/test skupu
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")
     
sc_train_aac = aac_df.loc[entries_sc_train].values
sc_test_aac = aac_df.loc[entries_sc_test].values

print(f"AAC-CV train: {sc_train_aac.shape} | test: {sc_test_aac.shape}")'''

'# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)\n# Selektujemo redove koji pripadaju SC train/test skupu\naac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")\n     \nsc_train_aac = aac_df.loc[entries_sc_train].values\nsc_test_aac = aac_df.loc[entries_sc_test].values\n\nprint(f"AAC-CV train: {sc_train_aac.shape} | test: {sc_test_aac.shape}")'

### TF-IDF + SVD i Fizičko-hemijske osobine

In [ ]:
'''# Fitovano na SC train skupu u 03
sc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"))
sc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"))

# Fizicko-hemijske osobine, fitovane na SC train skupu u 03
sc_train_ph = np.load(os.path.join(FEATURES_DIR, "sc_train_physchem_scaled.npy"))
sc_test_ph = np.load(os.path.join(FEATURES_DIR, "sc_test_physchem_scaled.npy"))

print(f"TF-IDF-SVD train: {sc_train_tfidf.shape} | test: {sc_test_tfidf.shape}")
print(f"PH train: {sc_train_ph.shape} | test: {sc_test_ph.shape}")'''

'# Fitovano na SC train skupu u 03\nsc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"))\nsc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"))\n\n# Fizicko-hemijske osobine, fitovane na SC train skupu u 03\nsc_train_ph = np.load(os.path.join(FEATURES_DIR, "sc_train_physchem_scaled.npy"))\nsc_test_ph = np.load(os.path.join(FEATURES_DIR, "sc_test_physchem_scaled.npy"))\n\nprint(f"TF-IDF-SVD train: {sc_train_tfidf.shape} | test: {sc_test_tfidf.shape}")\nprint(f"PH train: {sc_train_ph.shape} | test: {sc_test_ph.shape}")'

### Kombinovanje u feature setove

In [ ]:
'''feature_sets_train = {
    "AAC-CV": sc_train_aac,
    "TFIDF-SVD": sc_train_tfidf,
    "PH": sc_train_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([sc_train_aac, sc_train_tfidf]),
    "AAC-CV+PH": np.hstack([sc_train_aac, sc_train_ph]),
    "TFIDF-SVD+PH": np.hstack([sc_train_tfidf, sc_train_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([sc_train_aac, sc_train_tfidf, sc_train_ph]),
}

feature_sets_test = {
    "AAC-CV": sc_test_aac,
    "TFIDF-SVD": sc_test_tfidf,
    "PH": sc_test_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([sc_test_aac, sc_test_tfidf]),
    "AAC-CV+PH": np.hstack([sc_test_aac, sc_test_ph]),
    "TFIDF-SVD+PH": np.hstack([sc_test_tfidf, sc_test_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([sc_test_aac, sc_test_tfidf, sc_test_ph]),
}

for name, matrix in feature_sets_train.items():
    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")'''

'feature_sets_train = {\n    "AAC-CV": sc_train_aac,\n    "TFIDF-SVD": sc_train_tfidf,\n    "PH": sc_train_ph,\n    "AAC-CV+TFIDF-SVD": np.hstack([sc_train_aac, sc_train_tfidf]),\n    "AAC-CV+PH": np.hstack([sc_train_aac, sc_train_ph]),\n    "TFIDF-SVD+PH": np.hstack([sc_train_tfidf, sc_train_ph]),\n    "AAC-CV+TFIDF-SVD+PH": np.hstack([sc_train_aac, sc_train_tfidf, sc_train_ph]),\n}\n\nfeature_sets_test = {\n    "AAC-CV": sc_test_aac,\n    "TFIDF-SVD": sc_test_tfidf,\n    "PH": sc_test_ph,\n    "AAC-CV+TFIDF-SVD": np.hstack([sc_test_aac, sc_test_tfidf]),\n    "AAC-CV+PH": np.hstack([sc_test_aac, sc_test_ph]),\n    "TFIDF-SVD+PH": np.hstack([sc_test_tfidf, sc_test_ph]),\n    "AAC-CV+TFIDF-SVD+PH": np.hstack([sc_test_aac, sc_test_tfidf, sc_test_ph]),\n}\n\nfor name, matrix in feature_sets_train.items():\n    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")'

### Definisanje modela i mreže hiperparametara

In [8]:
param_grids = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.01, 0.1, 1, 10, 100],
            "solver": ["lbfgs"],
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "n_estimators": [200],
            "max_depth": [10, 15],
            "min_samples_leaf": [2, 5],
        },
    },
    "SVM": {
        "model": SVC(class_weight="balanced", random_state=RANDOM_STATE),
        "params": {
            "C": [0.1, 1, 10],
            "kernel": ["rbf", "linear"],
        },
    },
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [12]:
from joblib import Memory

cache_dir = "../cache/pipeline_cache"
memory = Memory(location=cache_dir, verbose=0)

all_results = []

for feature_name in FEATURE_SET_NAMES:
    preprocessor = make_preprocessor(feature_name)

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        full_pipeline = Pipeline([
            ("features", preprocessor),
            ("model", config["model"]),
        ], memory=memory)

        pipeline_param_grid = {
            f"model__{param_name}": values 
            for param_name, values in config["params"].items()
        }

        grid = GridSearchCV(
            estimator=full_pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            refit=True,
        )

        grid.fit(X_train_raw, y_sc_train)

        best_pipeline = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_pipeline.predict(X_train_raw)
        train_acc = accuracy_score(y_sc_train, y_train_pred)
        train_f1 = f1_score(y_sc_train, y_train_pred, average="weighted")

        # Predikcije na test skupu
        y_test_pred = best_pipeline.predict(X_test_raw)
        test_acc = accuracy_score(y_sc_test, y_test_pred)
        test_f1 = f1_score(y_sc_test, y_test_pred, average="weighted")

        best_params = {
            k.replace("model__", ""): v for k, v in grid.best_params_.items()
        }

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": best_params,

            "CV F1 (weighted)": grid.best_score_,

            "Train Accuracy": train_acc,
            "Train F1 (weighted)": train_f1,

            "Test Accuracy": test_acc,
            "Test F1 (weighted)": test_f1,
            
            "Overfit Gap (F1)": train_f1 - test_f1,
        })

        model_filename = f"sc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_pipeline, os.path.join(MODELS_DIR, model_filename))

    memory.clear()

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH


...

Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache



Treniranje završeno.


In [ ]:
'''all_results = []

for feature_name, X_train in feature_sets_train.items():
    X_test = feature_sets_test[feature_name]

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=cv,
            scoring="f1_weighted",
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_sc_train)

        best_model = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_model.predict(X_train)
        train_acc = accuracy_score(y_sc_train, y_train_pred)
        train_f1 = f1_score(y_sc_train, y_train_pred, average="weighted")

        # Predikcije na test skupu
        y_test_pred = best_model.predict(X_test)
        test_acc = accuracy_score(y_sc_test, y_test_pred)
        test_f1 = f1_score(y_sc_test, y_test_pred, average="weighted")

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 (weighted)": grid.best_score_,

            "Train Accuracy": train_acc,
            "Train F1 (weighted)": train_f1,

            "Test Accuracy": test_acc,
            "Test F1 (weighted)": test_f1,
            
            "Overfit Gap (F1)": train_f1 - test_f1,
        })

        model_filename = f"sc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_model, os.path.join(MODELS_DIR, model_filename))

print("\nTreniranje završeno.")'''

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH
Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: AAC-CV+PH
Treniranje: RandomForest         | Feature set: AAC-CV+PH
Treniranje: SVM                  | Feature set: AAC-CV+PH
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM  

### Pregled i čuvanje rezultata

In [13]:
results_df = pd.DataFrame(all_results)

# Sortiranje po Test F1, uz uvid u Overfit Gap
results_df = results_df.sort_values(by="CV F1 (weighted)", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(METRICS_DIR, "sc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train Accuracy", "Train F1 (weighted)",
    "CV F1 (weighted)",
    "Test Accuracy", "Test F1 (weighted)",
    "Overfit Gap (F1)",
]

results_df[display_cols].style.format({
    "Train Accuracy": "{:.3f}",
    "Train F1 (weighted)": "{:.3f}",
    "CV F1 (weighted)": "{:.3f}",
    "Test Accuracy": "{:.3f}",
    "Test F1 (weighted)": "{:.3f}",
    "Overfit Gap (F1)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1)"], cmap="Reds")

,Model,Feature set,Train Accuracy,Train F1 (weighted),CV F1 (weighted),Test Accuracy,Test F1 (weighted),Overfit Gap (F1)
0,SVM,AAC-CV+TFIDF-SVD,0.963,0.963,0.701,0.715,0.718,0.245
1,SVM,TFIDF-SVD,0.972,0.972,0.698,0.714,0.717,0.255
2,SVM,AAC-CV+TFIDF-SVD+PH,0.819,0.821,0.685,0.679,0.685,0.135
3,SVM,TFIDF-SVD+PH,0.817,0.819,0.683,0.680,0.687,0.132
4,RandomForest,AAC-CV+TFIDF-SVD+PH,0.955,0.955,0.667,0.668,0.670,0.285
5,LogisticRegression,AAC-CV+TFIDF-SVD+PH,0.689,0.694,0.664,0.641,0.649,0.045
6,RandomForest,TFIDF-SVD+PH,0.953,0.953,0.660,0.670,0.671,0.282
7,LogisticRegression,TFIDF-SVD+PH,0.687,0.692,0.660,0.642,0.649,0.043
8,RandomForest,AAC-CV+PH,0.976,0.976,0.656,0.681,0.682,0.294
9,LogisticRegression,AAC-CV+TFIDF-SVD,0.679,0.685,0.653,0.640,0.648,0.037
